<a href="https://colab.research.google.com/github/RuiRodrigues-lab/DataScienceFE/blob/Locker/CP4_3(TH).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.diagnostic import lilliefors
from scipy.stats import wilcoxon
import rpy2.robjects as ro
from scipy.stats import ttest_ind #independente
from scipy.stats import ttest_rel, t #emparelhados

#Precisamos desta biblioteca para podermos escolher um ficheiro local
#Se o ficheiro vier por API ou tivermos um link, é so alterar a forma de import
from google.colab import files

# 1️⃣ Faz upload do ficheiro (vai abrir uma janela para escolher no teu PC)
uploaded = files.upload()

# 2️⃣ Guarda o nome do ficheiro (Colab mostra o nome depois do upload)
filename = list(uploaded.keys())[0]

# 3️⃣ Lê o Excel, por default lê sempre a primeira tab, por isso podemos usar o "sheet_name"
# Se tivermos dados em varias tabs, devemos usar uma Dataframe(df) para cada uma das tabs
df = pd.read_excel(filename, sheet_name='Exerc3')
df.head()

Saving CP4.xlsx to CP4 (3).xlsx


,Vendas,Campanha
0,190,Antes
1,170,Antes
2,280,Antes
3,225,Antes
4,190,Antes


In [23]:
n_total = len(df["Vendas"])
n_non_missing = df["Vendas"].notna().sum()

print("Total:", n_total)
print("Não-missing:", n_non_missing)

Total: 66
Não-missing: 66


In [24]:
# 2) Separar por grupos
df["Campanha"] = pd.Categorical(df["Campanha"], categories=["Antes", "Depois"])

# 3) Criar grupos A e B
AntesA = df.loc[df["Campanha"] == "Antes", "Vendas"].dropna()
DepoisB = df.loc[df["Campanha"] == "Depois", "Vendas"].dropna()

print("\nlength(Antes) =", len(AntesA))
print("length(Depois) =", len(DepoisB))


length(Antes) = 33
length(Depois) = 33


In [25]:
# 4) Teste de normalidade Lilliefors (Kolmogorov-Smirnov com correcção) para cada grupo
D_A, p_A = lilliefors(AntesA, dist='norm')
D_B, p_B = lilliefors(DepoisB, dist='norm')

print("\nLilliefors (grupo A):")
print("D =", D_A)
print("p-value =", p_A)

print("\nLilliefors (grupo B):")
print("D =", D_B)
print("p-value =", p_B)


Lilliefors (grupo A):
D = 0.0870706375111428
p-value = 0.7585064392459729

Lilliefors (grupo B):
D = 0.09295290883343937
p-value = 0.6638981408042235


In [26]:
# teste emparelhado: H1: Antes < Depois
t_stat, p_two_sided = ttest_rel(AntesA, DepoisB)

# converter para unilateral H1: mu_Antes < mu_Depois
if t_stat < 0:
    p_value = p_two_sided / 2
else:
    p_value = 1 - p_two_sided / 2

print("Paired t-test (Antes < Depois)")
print("t =", t_stat)
print("p-value =", p_value)

Paired t-test (Antes < Depois)
t = -4.165085174584935
p-value = 0.00010988314918758822


In [28]:
# Teste t emparelhado (bilateral)
t_stat, p_two_sided = ttest_rel(grupoAntes, grupoDepois)

# Converter para unilateral H1: mean(Antes) < mean(Depois)
if t_stat < 0:
    p_value = p_two_sided / 2
else:
    p_value = 1 - p_two_sided / 2

print("Paired t-test (Antes < Depois)")
print("t =", t_stat)
print("df =", len(grupoAntes) - 1)
print("p-value =", p_value)

# ---- Intervalo unilateral 95% ----
dif = grupoAntes - grupoDepois
mean_diff = dif.mean()
sd_diff = dif.std(ddof=1)
n = len(dif)

tcrit = t.ppf(0.95, n - 1)  # one-sided, upper 95% (because CI is -Inf to ...)
ci_upper = mean_diff + tcrit * sd_diff / np.sqrt(n)


Paired t-test (Antes < Depois)
t = -4.165085174584935
df = 32
p-value = 0.00010988314918758822
